In [3]:
# packages used for creating and geo-locating the graph
import networkx as nx
import shapely

# packages needed for inspecting the output
import pandas as pd
import geopandas as gpd
import opentnsim.fis as fis

# packages needed for plotting
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# plot libraries
import folium

import numpy as np

In [4]:
# load the processed version from the Fairway Information System graph provided by Rijkswaterstaat
# For FIS version 0.3 
# FG = fis.load_network(version="0.3")

# Trying out for EuRIS
FG = fis.load_network(network="euris", version="0.1")

In [7]:
# Relevant polygon
poly_coords = [
    (3.3215331, 52.1076298),
    (3.2318116, 51.1184662),
    (5.0115969, 51.0782204),
    (5.0427246, 52.0789434),
    (3.3215331, 52.1076298)
]

polygon = shapely.Polygon(poly_coords)


In [10]:
# Printing the data per edge to see what the names are. Check if there is water depth info
for u, v, data in FG.edges(data=True):
    print (u, "->", v)
    print(data)
    break

BE_J8802 -> BE_J8803
{'name': 'Doksluis', 'name_cb': None, 'cntrycode': 'BE', 'cntrycode_cb': None, 'fw_code': '1K101', 'fw_code_cb': None, 'seq_nr': None, 'seq_nr_cb': None, 'code_cb': None, 'ww_name': 'Demeydokken', 'ww_name_cb': None, 'rt_name': 'Demeydokken', 'rt_name_cb': None, 'wwauthorit': 'Haven van Oostende', 'wwauthorit_cb': None, 'cemt': 'Va', 'mdraughtcm': 500.0, 'mlengthcm': 10500.0, 'mlencon': 10500, 'mwidthcm': 1680.0, 'mwidcon': 1680.0, 'speed': 'minder dan 2,5m diepgang: 11 km/u; vanaf 2,5m diepgang: 7,5 km/u', 'speedcon': 'minder dan 2,5m diepgang: 11 km/u; vanaf 2,5m diepgang: 7,5 km/u', 'calspeed_up': 7.5, 'calspeed_down': 7.5, 'calspeedc_up': 7.5, 'calspeedc_down': 7.5, 'maxspeed_up': nan, 'maxspeed_down': nan, 'maxspeedc_up': nan, 'maxspeedc_down': nan, 'tidedep': 0, 'tot_length': nan, 'estuary': nan, 'active': 1, 'ww_charges': nan, 'remark': None, 'istentec': nan, 'code': 'BE1K101', 'path': 'FairwaySection_BE_20250818.geojson', 'geometry': <LINESTRING (2.952 51.2

In [11]:
# PRINT FIS WATERDEPTH INFO 

# Create a map (pick a central point from your network)
m = folium.Map(location=[51.83, 4.33], zoom_start=10, tiles="cartodb positron")

for u, v, data in FG.edges(data=True):
    edge_geom = data["geometry"]  # This is already a Shapely LineString

    # Skip edges outside the polygon
    if not polygon.intersects(edge_geom):
        continue

    # Extract coordinates 
    points_x = list(data["geometry"].coords.xy[0])
    points_y = list(data["geometry"].coords.xy[1])
    line = [(points_y[i], points_x[i]) for i in range(len(points_x))]

    # Get depth value 
    # FIS
    # depth = data.get("GeneralDepth", None)
    # EuRIS
    depth_cm = data.get("mdraughtcm", None)

    # Convert to meters
    if depth_cm is None or (isinstance(depth_cm, float) and np.isnan(depth_cm)):
        depth = None
    else:
        depth = depth_cm / 100.0


    # Choose color based on depth
    if depth is None or (isinstance(depth, float) and np.isnan(depth)):
        color = "gray"
    else:
        if depth < 3:
            color = "red"
        elif depth < 6:
            color = "orange"
        else:
            color = "blue"

    # Add to map
    folium.PolyLine(
        line,
        color=color,
        weight=3,
        popup=f"Depth: {depth}"
    ).add_to(m)

# Display map
m